# Data Cleaning - Property Listings

Goal: load `raw_data.csv`, run a full data quality checklist, and export a validated `clean_data.csv`. Each section states what is being checked, why, and the conclusion.

In [24]:
import pandas as pd

## 1. Load and inspect data

In [25]:
# Load and inspect data
df = pd.read_csv('../data/raw_data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Property_ID    300 non-null    str  
 1   Area           300 non-null    int64
 2   Bedrooms       300 non-null    int64
 3   Bathrooms      300 non-null    int64
 4   Age            300 non-null    int64
 5   Location       300 non-null    str  
 6   Property_Type  300 non-null    str  
 7   Price          300 non-null    int64
dtypes: int64(5), str(3)
memory usage: 18.9 KB


## 2. Missing values
Check whether any column has empty/null entries — nulls can break downstream calculations or models if not handled.

In [26]:
# Checking if any of the columns have got empty values
df.isnull().sum()

Property_ID      0
Area             0
Bedrooms         0
Bathrooms        0
Age              0
Location         0
Property_Type    0
Price            0
dtype: int64

## 3. Duplicate rows
Check whether any full row is repeated — duplicate rows can silently bias any aggregation or model trained on this data.

In [27]:
# Checking if any row is having duplicates
df.duplicated().sum()

np.int64(0)

## 4. Duplicate IDs
`Property_ID` should be unique per listing. Checking this separately from full-row duplicates

In [28]:
# Checking if any of the Property ID is used twice
df['Property_ID'].duplicated().sum()

np.int64(0)

## 5. Shape, data types, and summary statistics
Confirms row/column counts, that numeric columns are actually stored as numbers (not text), and gives a sense of the range and distribution of each numeric field.

In [29]:
print(df.shape)
print(df.dtypes)
df.describe()

(300, 8)
Property_ID        str
Area             int64
Bedrooms         int64
Bathrooms        int64
Age              int64
Location           str
Property_Type      str
Price            int64
dtype: object


,Area,Bedrooms,Bathrooms,Age,Price
count,300.00000,300.000000,300.000000,300.000000,3.000000e+02
mean,2759.70000,3.033333,2.026667,25.000000,2.488366e+07
std,1297.68143,1.467219,0.792495,14.332646,1.266525e+07
min,520.00000,1.000000,1.000000,0.000000,3.695000e+06
25%,1675.75000,2.000000,1.000000,12.000000,1.527750e+07
50%,2738.00000,3.000000,2.000000,25.500000,2.236500e+07
75%,3801.25000,4.000000,3.000000,36.250000,3.460812e+07
max,4999.00000,5.000000,3.000000,49.000000,5.870000e+07


## 6. Categorical label consistency
Text columns are prone to inconsistent entries (casing, whitespace, spelling variants) that `describe()` won't reveal. Listing the unique values makes these visible.

In [30]:
print(df['Location'].unique())
print(df['Property_Type'].unique())

<StringArray>
['Rural', 'Suburb', 'City Center']
Length: 3, dtype: str
<StringArray>
['House', 'Villa', 'Apartment']
Length: 3, dtype: str


## 7. Range and logic validity
Some values are only invalid in context: negative area, negative price, or negative age don't make sense for a property listing even though they'd pass a simple type check.

In [31]:
print('Negative Area:', (df['Area'] < 0).sum())
print('Negative Price:', (df['Price'] < 0).sum())
print('Negative Age:', (df['Age'] < 0).sum())

Negative Area: 0
Negative Price: 0
Negative Age: 0


## 8. Outlier detection (IQR method)
Flags values that are valid but statistically extreme, using 1.5×IQR bounds on each numeric column. These aren't automatically removed — they're surfaced for review, since an extreme value can be a genuine luxury listing rather than an error.

In [32]:
for col in ['Area', 'Age', 'Price']:
    q1, q3 = df[col].quantile(.25), df[col].quantile(.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df[(df[col] < low) | (df[col] > high)]
    print(f'{col}: {len(outliers)} outliers (bounds: {low:.0f} to {high:.0f})')

Area: 0 outliers (bounds: -1512 to 6990)
Age: 0 outliers (bounds: -24 to 73)
Price: 0 outliers (bounds: -13718438 to 63604062)


## 9. Preparation and export
No corrective cleaning was required based on the checks above. As a preparation step, `Location` and `Property_Type` are cast to `category` dtype for memory efficiency and to guard against inconsistent entries being added later. The result is exported as the validated dataset for downstream analysis.

In [33]:
df['Location'] = df['Location'].astype('category')
df['Property_Type'] = df['Property_Type'].astype('category')

df.to_csv('../data/clean_data.csv', index=False)
print('Saved clean_data.csv:', df.shape)

Saved clean_data.csv: (300, 8)
